# Import Tendencies

In [13]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pickle
from google.colab import files
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import LabelEncoder

In [2]:
!pip install -q kaggle

# Data Loading

In [5]:
# 1. Munculkan tombol upload untuk memilih file kaggle.json
print("Silakan upload file kaggle.json:")
files.upload()

# 2. Buat direktori kaggle, pindahkan file, dan atur permission-nya
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

print("Kaggle API token berhasil dikonfigurasi!")

Silakan upload file kaggle.json:


Saving kaggle.json to kaggle.json
Kaggle API token berhasil dikonfigurasi!


In [6]:
!kaggle datasets download -d ahmedabbas757/coffee-sales --unzip

Dataset URL: https://www.kaggle.com/datasets/ahmedabbas757/coffee-sales
License(s): GNU Lesser General Public License 3.0
100% 8.23M/8.23M [00:00<00:00, 64.8MB/s]



In [14]:
df = pd.read_excel('/content/Coffee Shop Sales.xlsx')
df['transaction_date'] = pd.to_datetime(df['transaction_date'])
df.head()

,transaction_id,transaction_date,transaction_time,transaction_qty,store_id,store_location,product_id,unit_price,product_category,product_type,product_detail
0,1,2023-01-01,07:06:11,2,5,Lower Manhattan,32,3.0,Coffee,Gourmet brewed coffee,Ethiopia Rg
1,2,2023-01-01,07:08:56,2,5,Lower Manhattan,57,3.1,Tea,Brewed Chai tea,Spicy Eye Opener Chai Lg
2,3,2023-01-01,07:14:04,2,5,Lower Manhattan,59,4.5,Drinking Chocolate,Hot chocolate,Dark chocolate Lg
3,4,2023-01-01,07:20:24,1,5,Lower Manhattan,22,2.0,Coffee,Drip coffee,Our Old Time Diner Blend Sm
4,5,2023-01-01,07:22:41,2,5,Lower Manhattan,57,3.1,Tea,Brewed Chai tea,Spicy Eye Opener Chai Lg


In [9]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 149116 entries, 0 to 149115
Data columns (total 11 columns):
 #   Column            Non-Null Count   Dtype         
---  ------            --------------   -----         
 0   transaction_id    149116 non-null  int64         
 1   transaction_date  149116 non-null  datetime64[ns]
 2   transaction_time  149116 non-null  object        
 3   transaction_qty   149116 non-null  int64         
 4   store_id          149116 non-null  int64         
 5   store_location    149116 non-null  object        
 6   product_id        149116 non-null  int64         
 7   unit_price        149116 non-null  float64       
 8   product_category  149116 non-null  object        
 9   product_type      149116 non-null  object        
 10  product_detail    149116 non-null  object        
dtypes: datetime64[ns](1), float64(1), int64(4), object(5)
memory usage: 12.5+ MB


In [15]:
# print("=== PRODUCT CATEGORY ===")
# print(df['product_category'].unique())

# print("\n=== PRODUCT TYPE per CATEGORY ===")
# print(df.groupby(['product_category', 'product_type'])['product_detail'].count().reset_index())

In [16]:
# kategori_pilih = ['Coffee', 'Tea', 'Drinking Chocolate', 'Bakery']
# df_filtered = df[df['product_category'].isin(kategori_pilih)]

# # Hitung total qty per product_detail per kategori
# top_menu = (
#     df_filtered.groupby(['product_category', 'product_detail'])['transaction_qty']
#     .sum()
#     .reset_index()
#     .sort_values(['product_category', 'transaction_qty'], ascending=[True, False])
# )

# # Ambil top 3 per kategori
# top3 = top_menu.groupby('product_category').head(3)
# print(top3.to_string(index=False))

In [17]:
# Filter 12 menu terpilih
menu_pilih = [
    'Chocolate Croissant', 'Ginger Scone', 'Cranberry Scone',
    'Latte', 'Columbian Medium Roast Rg', 'Latte Rg',
    'Dark chocolate Lg', 'Sustainably Grown Organic Lg', 'Sustainably Grown Organic Rg',
    'Earl Grey Rg', 'Morning Sunrise Chai Rg', 'Peppermint Rg'
]
cabang_pilih = ['Astoria', "Hell's Kitchen", 'Lower Manhattan']

In [19]:
df_filtered = df[
    (df['product_detail'].isin(menu_pilih)) &
    (df['store_location'].isin(cabang_pilih))
]

In [20]:
# ── 2. Agregasi harian per menu per cabang ──
df_agg = (
    df_filtered.groupby(['transaction_date', 'store_location', 'product_detail'])['transaction_qty']
    .sum()
    .reset_index()
    .rename(columns={'transaction_qty': 'total_qty'})
)

In [21]:
# ── 3. Training 36 model ──
models = {}

for cabang in cabang_pilih:
    models[cabang] = {}
    df_cabang = df_agg[df_agg['store_location'] == cabang]

    for menu in menu_pilih:
        df_menu = df_cabang[df_cabang['product_detail'] == menu].copy()

        if len(df_menu) < 7:
            print(f"SKIP: {cabang} - {menu} (data kurang)")
            continue

        # Feature: day of week + day index sebagai trend
        df_menu = df_menu.sort_values('transaction_date')
        df_menu['day_index'] = range(len(df_menu))
        df_menu['day_of_week'] = df_menu['transaction_date'].dt.dayofweek

        X = df_menu[['day_index', 'day_of_week']].values
        y = df_menu['total_qty'].values

        model = LinearRegression()
        model.fit(X, y)

        models[cabang][menu] = {
            'model': model,
            'last_day_index': df_menu['day_index'].max(),
            'last_date': df_menu['transaction_date'].max()
        }
        print(f"✓ Trained: {cabang} - {menu}")

✓ Trained: Astoria - Chocolate Croissant
✓ Trained: Astoria - Ginger Scone
✓ Trained: Astoria - Cranberry Scone
✓ Trained: Astoria - Latte
✓ Trained: Astoria - Columbian Medium Roast Rg
✓ Trained: Astoria - Latte Rg
✓ Trained: Astoria - Dark chocolate Lg
✓ Trained: Astoria - Sustainably Grown Organic Lg
✓ Trained: Astoria - Sustainably Grown Organic Rg
✓ Trained: Astoria - Earl Grey Rg
✓ Trained: Astoria - Morning Sunrise Chai Rg
✓ Trained: Astoria - Peppermint Rg
✓ Trained: Hell's Kitchen - Chocolate Croissant
✓ Trained: Hell's Kitchen - Ginger Scone
✓ Trained: Hell's Kitchen - Cranberry Scone
✓ Trained: Hell's Kitchen - Latte
✓ Trained: Hell's Kitchen - Columbian Medium Roast Rg
✓ Trained: Hell's Kitchen - Latte Rg
✓ Trained: Hell's Kitchen - Dark chocolate Lg
✓ Trained: Hell's Kitchen - Sustainably Grown Organic Lg
✓ Trained: Hell's Kitchen - Sustainably Grown Organic Rg
✓ Trained: Hell's Kitchen - Earl Grey Rg
✓ Trained: Hell's Kitchen - Morning Sunrise Chai Rg
✓ Trained: Hell's Ki

In [22]:
# ── 4. Simpan ke model.pkl ──
with open('model.pkl', 'wb') as f:
    pickle.dump(models, f)

print("\n✅ model.pkl berhasil disimpan!")
print(f"Total model: {sum(len(v) for v in models.values())}")


✅ model.pkl berhasil disimpan!
Total model: 36
